# Ground State Preparation of Molecules Using VQE and GQE
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$


In [ ]:
!pip install cudaq -q
!pip install cudaq-solvers -q
## Note: cudaq-solvers requires libgfortran. If you see an ImportError, run:
## !apt-get install -y libgfortran5
!pip install cudaq-solvers[gqe] -q
!pip uninstall cupy-cuda12x -q -y
!wget -q https://github.com/osbama/KBM608/archive/refs/heads/main.zip
!unzip -q main.zip
!rm -rf Images aux_files
!mv KBM608-main/hands-on/hands-on-4-images/ ./Images
!mv KBM608-main/hands-on/hands-on-4-aux/ ./aux_files
!rm -rf KBM608-main
!rm -rf main.zip

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
from scipy.optimize import minimize

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import Image, display

import cudaq
from cudaq import spin

import cudaq_solvers as solvers

import time
from typing import List


## To install cudaq-solvers (if not already installed), uncomment and run:
## !pip install cudaq-solvers -q
## Note: cudaq-solvers requires libgfortran. If you see an ImportError, run:
## !apt-get install -y libgfortran5
import cudaq_solvers as solvers

#Disable this if you don't have GPUs
cudaq.set_target('nvidia', option='mqpu,fp64')

---
## The Ground State Problem

A fundamental quantity in quantum chemistry is the expectation value determined from a Hamiltonian operator $H$ and a quantum state (wavefunction) $\ket{\Psi}$.  The expectation value is defined as the average energy of the state $\ket{\Psi}$ and is represented with the following expression.

$$E =  \bra{\Psi}H\ket{\Psi}$$

This energy can be computed for any arbitrary state, but is often most important when obtained from a particular state called the **ground state**. The ground state is defined as the state which produces the lowest expectation value.

For chemists, the ground state energy of a molecule provides a wealth of knowledge about its reactivity and chemical properties as most molecules are in the ground state rather than excited states. For quantum computing more broadly, non-physics optimization problems can often be mapped to a Hamiltonian. The ground state of such a Hamiltonian, if obtained, can be sampled to produce high-quality solutions to the original optimization problem.  

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/pes.png?raw=1" alt="Potential energy surface plot showing molecular energy as a function of bond distance, with the minimum corresponding to the equilibrium geometry and ground state energy." width="600">
  <figcaption>Accurate ground state energies are necessary to compute the strength of a chemical bond and other important properties. </figcaption>
</figure>

Identifying and preparing the gound state is an incredible challenge and is necessary to obtain quantities like the ground state energy. Consider an arbitrary quantum state constructed from $N$ qubits which is defined by a $2^N$ state vector of complex entries called amplitudes.  These amplitudes can take any complex value as long as they remain normalized.  

$$
\ket{\psi} =
\begin{pmatrix}
\alpha_0 \\
\alpha_1 \\
\vdots \\
\alpha_{2^n-1}
\end{pmatrix}
\qquad
\text{with} \qquad
\sum_{k=0}^{2^n-1} \lvert\alpha_k\rvert^2 = 1
$$

The first issue arises from the exponential size of the quantum state.  Even if one knows all parameters of a given ground state exactly, it is impossible to store a complete state vector for systems of just over 50 qubits, even if we could pool the memory of every supercomputer in the world!

Quantum computing subverts this issue using superposition and instead representing quantum states as circuits where a sub-exponential number of gates can be applied to $N$ qubits. See the example below. A 16 element state vector is constructed using only 8 gates applied to 4 qubits.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/sv.png?raw=1" alt="Diagram showing a 4-qubit quantum circuit with 8 single-qubit gates on the left, and the corresponding 16-element state vector on the right, illustrating how a sub-exponential number of gates encodes an exponentially large quantum state." width="600">
</figure>



### Exercise 1:

Calculate the number of elements in the state vector and the number of gates required to prepare the same uniform superposition state for a system with 10 qubits and a system with 20 qubits.


Solution:
N = 10 -> 20 gates -> 1024 element state vector;
N = 20 -> 40 gates -> 1,048,576 element state vector

The second issue pertains to actually preparing a ground state with a quantum circuit. There is an infinite number of possible gate combinations and, without any intuition, blindly preparing a ground state circuit might be just as challenging as the original problem.  Cases like chemistry make this a bit easier because the underlying physics provides some structure , but finding the exact ground state may still require a prohibitively large number of gate operations for practical purposes.

This is where approximate methods are important for preparing states that are "close enough" to the ground state.


---
## Variational Methods and their Problems

Thanks to the **variational theorem**, the search for a circuit that approximates the ground state can become a bit more systematic. The variational theorem states that for any normalized trial wavefunction $\ket{\Psi_{trial}}$ that

$$\bra{\Psi_{trial}}H\ket{\Psi_{trial}} \ge E_0, $$

where $E_0$ is the ground state energy. This means that any trial wavefunction will produce an energy that is an upper bound to the ground state energy. In other words, we can try any trial wavefunction and the lowest energy result is the best approximation of the ground state.  This at least provides a way to rank our trial wavefunctions.

This fact has inspired an entire class of quantum algorithms called **variational algorithms** which take advantage of the strengths of both quantum and classical processors working in tandem. The image below depicts the workflow of a general variational algorithm.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/variational-diagram.png?raw=1" alt="Diagram of the variational quantum algorithm workflow: a parameterized quantum circuit (**ansatz**) with rotation gates is prepared, the expectation value is measured on a QPU, a classical optimizer updates the parameters, and the cycle repeats until convergence." width="900">

Th steps are as follows:

1. A predetermined structure or **(ansatz)** of a circuit is chosen where some of the rotation gates are parameterized ($\Theta={\theta_0 \cdots \theta_n}$).
2. The parameters are then randomly initialized (i.e., $\Theta$ is set to $\Theta_0$).
3. An expectation value is computed on a quantum computer using the $\bra{\Psi(\Theta)}H\ket{\Psi(\Theta)}$.
4. The resulting energy is fed into a classical optimizer which updates the parameter values $\Theta$.
5. Steps 3 and 4 are repeated until convergence.



Despite the ingenuity behind variational algorithms, there are many problems that limit their effectiveness.  One of the most documented is the so-called **barren plateau** problem.  For even small cases, like the 14-qubit example taken from ["Quantum variational algorithms are swamped with traps"](https://www.nature.com/articles/s41467-022-35364-5) shown below, the loss function landscape is extremely challenging, and, at problem sizes of even moderate size, the landscape usually becomes too flat to optimize.

 <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/surface.png?raw=1" alt="3D loss function landscape for a 14-qubit VQE example showing an extremely flat, featureless surface with only tiny variations, illustrating the barren plateau problem where gradients vanish and optimization fails." width="900">

We have also assumed evaluation of $\bra{\Psi_{trial}}H\ket{\Psi_{trial}}$ is efficient on a QPU in the sense that it is possible to obtain expectation values for states much larger than ever possible with exact classical methods.  However, the wallclock time to run these on a QPU may still be quite costly and produce a practical limitation for VQE.  The barren plateau problem makes this problem worse as larger molecule require many more steps to converge the energy, if it will converge at all.  Gradients can help, but are extremely costly to evaluate in their own right.

Consider the image below.  $H_2$ quickly converges as it has only three parameters, but even a small molecule like $N_2$ has 609 parameters and is far from converged after 300 iterations.  Considering the exact energy of these molecules can easily be computed exactly with matrix diagonalization and convergence issues are already becoming apparent means systems that require large number of qubits quickly become prohibitive to solve.



### Exercise 2:

Use the CUDA-Q Solvers code below to run VQE for $H_2$, $LiH$, and $N_2$ with a standard unitary coupled cluster with singled and double excitations (UCCSD) ansatz.  Note how many parameters are required for each system and comment on the convergence behavior.   Note, that these are still very small molecules with small basis sets.  Try playing around with the function $\texttt{stateprep.get\_num\_uccsd\_parameters}$ with different qubit counts and number of electrons to see how parameter numbers can quickly get out of hand.





In [ ]:
# EXERCISE 2

# Create the molecular hamiltonian
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., 0.74))]
#geometry = [('Li', (0., 0., 0.)), ('H', (0., 0., 1.59))]
#geometry = [('N', (0., 0., 0.)), ('N', (0., 0., 1.10))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, CASCI = True)


# Get the number of qubits and electrons
numQubits = molecule.n_orbitals * 2
numElectrons = molecule.n_electrons

spin = 0
initialX = [-.2] * solvers.stateprep.get_num_uccsd_parameters(
    numElectrons, numQubits)


# Define the UCCSD ansatz
@cudaq.kernel
def ansatz(thetas: list[float]):
    q = cudaq.qvector(numQubits)
    for i in range(numElectrons):
        x(q[i])
    solvers.stateprep.uccsd(q, thetas, numElectrons, spin)


def cost(theta: list[float]) -> float:

    exp_val = cudaq.observe(ansatz, molecule.hamiltonian, theta).expectation()

    return exp_val

exp_vals = []


def callback(xk: list[float]) -> None:
    exp_vals.append(cost(xk))


cudaq.set_target("nvidia", option = 'fp64')
result = minimize(cost,
                  initialX,
                  method='COBYLA',
                  callback=callback,
                  options={'maxiter': 300})

print('UCCSD-VQE energy =  ', result.fun)
print('Number of Variational Parameters =', len(initialX))

plt.plot(exp_vals)
plt.xlabel('Epochs')
plt.ylabel('Energy')
plt.title('VQE')
plt.show()


print(cudaq.sample(ansatz, result.x))

Another issue is ansatz selection. The quality of the result is limited by how expressive the selected ansatz is. In the case of $H_2$, the ground state can be obtained exactly, because the **UCCSD** ansatz provides sufficient entanglement structure to match the exact full configuration interaction (FCI) result.

This is not necessarily true if we select some other ansatz.  



### Exercise 3:

Use the code below to specify a custom ansatz in the kernel below in the following manner. Apply $H$ gates to the two occupied qubits.  Add CNOT gates between each adjacent qubit pair and then apply a parameterized $R_X$, $R_Y$, and $R_Z$ to each qubit.  This should result in a VQE with 12 parameters, 6 times more than UCCSD.  Comment on the convergence of the problem.  How does the final energy compare to the FCI energy? Why might this be the result? Hint: Try running $\texttt{cudaq.sample()}$ for the converged UCCSD result and for your ansatz.


In [ ]:
# EXERCISE 3
cudaq.set_target("nvidia", option = 'fp64')
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., 0.74))]

molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, casci = True)

print(molecule.energies)

# Get the number of qubits and electrons
numQubits = molecule.n_orbitals * 2
numElectrons = molecule.n_electrons

spin = 0
initialX = [-.3] * 12

@cudaq.kernel
def custom_ansatz(thetas: list[float]):
    q = cudaq.qvector(numQubits)

    x(q[0]) #prepares Hartree Fock State
    x(q[1])

    h(q[0]) #prepares Hartree Fock State
    h(q[1])

    ##TODO## build the circuit: CNOT between adjacent qubit pairs, then Rx/Ry/Rz on each qubit


print(cudaq.draw(custom_ansatz, initialX))


def cost(theta: list[float]) -> float:

    exp_val = cudaq.observe(custom_ansatz, molecule.hamiltonian, theta).expectation()

    return exp_val

exp_vals = []


def callback(xk: list[float]) -> None:
    exp_vals.append(cost(xk))

cudaq.set_target('nvidia')
result = minimize(cost,
                  initialX,
                  method='COBYLA',
                  callback=callback,
                  options={'maxiter': 300})

print('Custom Ansatz Energy =  ', result.fun)
print('Number of Variational Parameters =', len(initialX))
print('Variational Parameters =', result.x)

print(cudaq.draw(custom_ansatz, result.x))

plt.plot(exp_vals)
plt.xlabel('Epochs')
plt.ylabel('Energy')
plt.title('VQE')
plt.show()

##TODO##
print(cudaq.sample(custom_ansatz, result.x)) #does not preserve particle number

Chemistry provides some good intuition for ansatz selection, but this starts to break down with larger, strongly correlated molecules.

---
## The Generative Quantum Eigensolver

In an attempt to address the issues posed by VQE for finding molecular ground states, researchers from NVIDIA, U. Toronto, and St Jude Children's Research Hospital developed a technique called the generative quantum eigensolver (GQE) which uses a **generative pretrained transformer (GPT)** model to sample circuits which approximate a molecule's ground state.

This section will explain some of the technical details of GQE, but first consider how this compares to VQE at a high level. The diagram below shows the two workflows.

 <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/gqe_vqe.png?raw=1" alt="Side-by-side workflow comparison of VQE (left) and GQE (right): both prepare a quantum circuit and measure energy, but VQE updates circuit rotation parameters while GQE updates neural network (GPT model) parameters to sample better circuits." width="900">
Notice that both are similar in the sense that a quantum circuit is prepared and sampled to produce an energy which informs how the model parameters are updated.  The key difference is that with GQE, the optimized parameters are the neural network parameters of the GPT model and not within the quantum circuits.  This provides a much better optimization landscape, avoiding the **barren plateau** issue. There are further benefits related to pretraining, but these will be discussed later.

Let's zoom in a bit on how the GPT model works and circuits are sampled.  First, the model needs a vocabulary.  The natural choice is a set of time evolution operators derived from a chemically inspired ansatz like UCCSD.  In the paper, the authors have a large set of operators of the form $e^{iP_jt_k}$ where $P_j$ is an operator from UCCSD and  $t_k$ is some a time step selected from a range of possible time steps.

Just as ansatz selection for VQE was critical for convergence, a good vocabulary is key for GQE.  GQE just needs to be given an vocabulary that is flexible enough. It is also possible to select a vocabulary tuned to specific qubit modalities such as those with nearest neighbor connectivity.


The sampling process is shown in the figure below.  The GPT model takes the parameters, the vocabulary, an a partially constructed circuits (initially an empty circuit) as inputs.  The GPT model then outputs a logit vector related to the probability of sampling index $i$ corresponding to the $i$-th unitary in the vocabulary.  This index is then appended to the string and fed back into the GPT model until a sequence of $N$ (user specified) indices is created, i.e. the full circuit.


 <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/gqe_sample.png?raw=1" alt="Diagram of the GQE circuit sampling process: the GPT model receives the vocabulary (UCCSD operators), model parameters, and a partially built circuit as input, outputs a logit vector over operator indices, samples the next operator, and appends it to build the full circuit token by token." width="900">
This is the same process as a large language model (LLM) building a sentence.  If the input sentence is "The apple is colored..." A logit would be produced that would sample the token "red" with high probability.  The GQE model similarly learns which unitaries help construct a circuit close to the ground state.







### Exercise 4:

In the cell below, assume that the provided logit (w) was output from the GPT model.  Finish the function to compute the probability that each index $i=0$ to $i=9$ is sampled given an activation function of $e^{-\beta w_i}$ is used (like in the paper).  $\beta = \frac{1}{T}$ is called the inverse temperature as it is related to a Boltzmann distribution from statistical mechanics in this content.  Start with $\beta =1$.  What happens to the distribution when you change $\beta$?  Why might it be helpful to adjust beta during GQE sampling?

In [ ]:
# EXERCISE 4

def plot_boltzmann_distribution(w: list[float], beta: float) -> None:
    """
    Plots the energy vector w and the resulting probability distribution
    for a given inverse temperature beta.
    """

    ##TODO## compute Boltzmann probabilities: activation = exp(-beta * w_i), then normalize
    probabilities = None  ##TODO## replace with your computation
#--------------Setup Plotting-------------------------
    indices = np.arange(len(w))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    bars1 = ax1.bar(indices, w, color='skyblue', edgecolor='black')
    ax1.set_title(f'Logit Values', fontsize=14)
    ax1.set_xlabel('Index ($j$)', fontsize=12)
    ax1.set_ylabel('Value ($w_j$)', fontsize=12)
    ax1.set_xticks(indices)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                 f'{height:.1f}', ha='center', va='bottom')

    colors = ['coral' if p == max(probabilities) else 'lightgreen' for p in probabilities]
    bars2 = ax2.bar(indices, probabilities, color=colors, edgecolor='black')

    ax2.set_title(f'Probability Distribution ($\\beta={beta}$)', fontsize=14)
    ax2.set_xlabel('Index ($j$)', fontsize=12)
    ax2.set_ylabel('Probability $P(j) \\propto e^{-\\beta w_j}$', fontsize=12)
    ax2.set_xticks(indices)
    ax2.set_ylim(0, max(probabilities) * 1.15) # Add headroom for text
    ax2.grid(axis='y', linestyle='--', alpha=0.7)

    for bar in bars2:
        height = bar.get_height()
        if height > 0.001: # Only label if visible
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                     f'{height:.1%}', ha='center', va='bottom', fontweight='bold')

    plt.suptitle(f'Sampling with $\\beta={beta}$ (Inverse Temperature)', fontsize=16)
    plt.tight_layout()
    plt.show()

w = [1.0, 2.5, 0.5, 3.0, 1.5, 4.0, 0.2, 2.0, 5.0, 1.2]

plot_boltzmann_distribution(w,0.1)
plot_boltzmann_distribution(w,1)
plot_boltzmann_distribution(w,10)

Now you have an understanding of how circuits are constructed with the GQE model, and we can revisit the training procedure.  For each training Epoch, GQE will sample a set of $C$ circuits. A quantum computer is then required to sample each circuit to compute an expectation value.  Each of these evaluations can be done asynchronously on many QPUs in parallel, a workflow that CUDA-Q enables.

There are a number of ways to evaluate the cost function from this collection of energies including procedures called logit matching loss and group relative policy optimization-based loss, with the latter generally a better choice to avoid instances where the loss decreases but the energy plateaus. Curious readers can explore the details of these loss functions in the appendix of "The **Generative Quantum Eigensolver (GQE)** and its Application for Ground State Search".  




### Exercise 5:

This exercise will help you explore why GQE is more efficient and scalable compared to VQE. Consider the three plots below that demonstrate simulated comparison of GQE and VQE.  Comment on the accuracy of VQE vs GQE.  Now, complete the tables below to compare the quantum resources used between the two methods. Any user inputs are provided for you, and remember you can use functions from solvers to compute the number of UCCSD parameters.  Is there a benefit in the number of circuits run?  Are the circuits simpler?
         <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/vqe_vs_gqe.png?raw=1" alt="Three bar charts comparing VQE and GQE ground state energy accuracy for BeH2, N2, and CO2, showing GQE achieves similar or better accuracy using far fewer circuit evaluations." width="1300">




| Molecule | BeH$_2$ | N$_2$ | CO$_2$ |
| :--- | :---: | :---: | :---: |
| **Active Space (e, o)** | (4,6) | (6,6) | (9,10) |
| **VQE Parameters (UCCSD)** |  92| 117| 560 |
| **VQE Epochs** |2000  | 3000| 10000 |
| **VQE Circuit Evals. per Epoch** | 184 |234 | 1120 |
 **Total VQE Circuit Evals.** | 368000 | 702000|11200000  |
| **GQE Epochs** | 500 |3000 | 5000 |
| **GQE Evals. per Epochs** | 50 |50 | 50 |
| **Total GQE Circuit Evals.** | 25000 | 150000| 250000 |
| **GQE Total Circuit Evals. (% of VQE)** | 7 % | 21 %| 2 % |


| Molecule | BeH$_2$ | N$_2$ | CO$_2$ |
| :--- | :---: | :---: | :---: |
| **Active Space (e, o)** | (4,6) | (6,6) | (9,10) |
| **VQE Operators (Parameters) per Circuit** | 92 | 117 |560  |
| **GQE Operators per Circuit** | 60 | 100| 200 |
| **GQE Operators (% of VQE)** | 65 % | 85 %| 36 % |



In [ ]:
# EXERCISE 5
print(solvers.stateprep.get_num_uccsd_parameters(4,12))
print(solvers.stateprep.get_num_uccsd_parameters(6,12))
print(solvers.stateprep.get_num_uccsd_parameters(10,18))

---
##  Running GQE with CUDA-Q Solvers


GQE can be used in CUDA-Q Solvers but requires an additional install `pip install cudaq-solvers[gqe] -q`.

Running the GQE method in CUDA-Q Solvers is simple and can be broken down into a few simple steps. First a molecular Hamiltonian is constructed just like you would do for VQE. Next, an operator pool is constructed. The built-in function `get_gqe_pauli_pool` takes a list of potential parameters and combines them with the base UCCSD operators to form a collection of different operators and evolution time steps.  Note, you can customize this if you want, but we will just use the built-in function.

Next, a cost function is defined which utilizes a CUDA-Q kernel to apply the selected operators.

Finally, configuration settings for the model are specified that determine its training behavior.  More on these later.

The model then runs using the following API call: `solvers.gqe(cost, op_pool, config=cfg)` which produces the lowest energy circuit found by the routine and its corresponding operators.

An entire code example can be found [here](https://nvidia.github.io/cudaqx/examples_rst/solvers/gqe.html) with all of these steps. Explore the script if you want to develop your own workflow using GQE.



### Exercise 6:

To understand the GQE results, you will run a script called `run_gqe_h1.py` which is based off of the main workflow above, but with a few additional flags you can run to explore different settings in this notebook and easily plot the results.

Each example below considers $H_2$ with a 6-31G basis set and a (2,3) active space. This is a simple molecule to make model training fast but also see some interesting trends.

Important: you will change parameters and observe different model behavior.  We have set a random seed so results are reproducible. The trends we comment on will vary from system to system so the point is not to draw universal conclusions but see some of the challenges and considerations that arise with the GQE method.


💻 Just a heads-up: The rest of this notebook is designed to be run on an environment with a GPU. If you don't have access to a GPU, feel free to run the cells under script calls to load the precomputed results. Enjoy learning! ⭐




In [ ]:
# EXERCISE 6
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-7 --temperature 5.0 --output_file 'baseline.png'
display(Image(filename='baseline.png', width=600))

A few things to notice.  First, the final energy produced by the routine may not correspond to any of the circuits sampled in the final epoch.  This is because every epoch, 5 circuits are sampled and their energies measured. The final result is the lowest energy found from all  50 * 5 circuits sampled, even if it was found in an earlier epoch! In the instance above, that is epoch 36.  

In theory, as the model trains, it samples better circuits, but the process is training a distribution so it is not garunteed the best solution will be sampled at any particular epoch. This can somewhat be observed as the samples get generally lower in energy.  The training loss function below shows that the model is indeed training and would probably get even better if we let it run longer.

Also note that the data is output in a folder called `gqe_logs` which creates a JSON file with all of the data. Each epoch produces sets of index values that looks like this.

$$[324, 205, 0, 324, 205, 135, 258, 221, 190, 205, 233, 146, 236, 324, 376, 31, 205, 135, 324, 461]$$

This specifies a circuit which is constructed by adding operator 324 then 205, and so on based on the operators in the vocabulary. Each circuit has an associated energy saved too.

A final observation, note how many parameters are trained, over 86 million!

### Exploring Sample Size

Sample size is another key choice. Consider how sample size will impact the training and results. What is the downside to a higher sample count?  Try running with only 2 samples and then with 30.

In [ ]:
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 2 --lr 1e-7 --temperature 5.0 --output_file 'sample_small.png'
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 100 --lr 1e-7 --temperature 5.0 --output_file 'sample_large.png'


In [ ]:
display(Image(filename='sample_small.png', width=700))
display(Image(filename='baseline.png', width=700))
display(Image(filename='sample_large.png', width=700))

The key observation here is the convergence of the loss function. Notice that with high samples, it convergences more smoothly compared to only two samples and results in a better overall result. This makes intuitive sense. If the goal is to train a distribution, more samples will always provide a better representation of what is being trained.  The tradeoff is that more samples require more QPU time. So users must account for the resources they have available for training. Look at the timings shown for the training steps above. Notice how much longer it took to obtain 30 samples.  If this was a much larger circuit with more operators this would take even longer.

Thankfully, the GQE method can be easily parallelized and run every sample asynchronously at the same time on a different QPU or GPU.


### Exploring Gate Count

You can also set the gate count to specify how many operators are sampled for each circuit. Run a couple of cases below with a small and large number of operators/gates.  What problems might occur with too many or too few?



In [ ]:
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 4 --num_samples 5 --lr 1e-7 --temperature 5.0 --output_file 'gates_small.png'
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 100 --num_samples 5 --lr 1e-7 --temperature 5.0 --output_file 'gates_large.png'


In [ ]:
display(Image(filename='gates_small.png', width=600))
display(Image(filename='baseline.png', width=600))
display(Image(filename='gates_large.png', width=600))

You might have thought that more gates would improve the results, but in this case, it is the opposite. It is true that more gates provide more flexibility, but it also becomes harder to learn.  Notice how the loss function is much less converged with 100 gates. This may improve with more iterations, but it also might mean that there is some overfitting going on and we do not require as many gates to construct a good wavefunction.

It might also be the case that our vocabulary is the limiting factor and changing the operator pool is the fix.

Note that for more complex systems, too few gates can easily be a problem. If the circuits do not have enough flexibility to describe the entanglement structure of the wavefunction, poor results are guaranteed.

### Exploring Temperature

We can explore what happens when we change some of the settings.  A few cautions though.  First, the model training and sampling process is stochastic so any change we make is not garunteed to repeat what we observe. Our goal is to make some general observations about the impacts of changing hyperparameters like temperature and number of samples.  For our purposes, a fixed seed is used so the results are reproducible.

First try increasing the temperature to 30.  What do you expect to see? Also try running a case which a much lower temperature of 0.5.

In [ ]:
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-7 --temperature 0.5 --output_file 'low_temp.png'
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-7 --temperature 30 --output_file 'high_temp.png'

In [ ]:
display(Image(filename='low_temp.png', width=600))
display(Image(filename='baseline.png', width=600))
display(Image(filename='high_temp.png', width=600))

The impact of temperature is slightly harder to connect to the result, but it can be observed in the plots.   Recall that the temperature tunes the probability that the model samples less likely operators when building the circuit.  Notice the spread of the samples. Clearly, they spread increases as the temperature is raised.  This is important for larger systems where the search space is enormous and there may be many local minima that yield poor solutions.  In the case of $H_2$ the system is simple so there is not as much impact from temperature on the result.

Underneath the hood, GQE uses an adaptive temperature schedule that balances this tradeoff to ensure the balance between quality results and exploration of the Hilbert space by slowly increasing the temperature in later epochs.

### Exploring Learning Rate

The learning rate is another key parameter that shows up in many AI applications and essentially controls how much the neural network parameters are adjusted each epoch.  Try running the default setting with a high and low learning rate.  What do you expect?


In [ ]:
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-9 --temperature 5 --output_file 'low_lr.png'
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-4 --temperature 5 --output_file 'high_lr.png'


In [ ]:
display(Image(filename='low_lr.png', width=600))
display(Image(filename='baseline.png', width=600))
display(Image(filename='high_lr.png', width=600))

Notice in the first plot, the learning rate is so small, the model does not really learn, and the cost goes up even! Essentially, the samples do not guide the training or will do so at such a slow rate it is impractical.  On the flip side, notice how a high learning rate sends massive shocks to the parameters resulting in some large spikes in the loss when higher energy circuits are sampled.  This can cause problems and even cause the model to overshoot the correct answer.  This is particularly a problem when the temperature is high, and the learning rate is high. Try running that case below.

In [ ]:
!python3 /content/aux_files/run_gqe_h2.py --max_iters 50 --ngates 20 --num_samples 5 --lr 1e-3 --temperature 30 --output_file 'high_lr_temp.png'

In [ ]:
display(Image(filename='high_lr_temp.png', width=600))

In this case the training is a mess as the model is far too sensitive to higher energy samples.

Learning rate is another careful balance of time and training quality.

---

## Conclusion

The ground state problem is fundamental for chemistry and more broadly quantum computing.  In this notebook you learned how a prototypical variational method like VQE can help solve the problem but comes with a number of its own issues.  Much research is dedicated to solving this problem with clever techniques like AI.

The GQE method is an excellent and pioneering example of an improved approach that can converge with fewer quantum circuit executions, can run in parallel, and result in a pretrained model that can be transferred to other contexts.   Current work is exploring how GQE can be trained on chemical system A and then sample circuits for system B with little or no additional training.  

Likewise, GQE has already been applied to other domains like [combinatorial optimization](https://arxiv.org/abs/2501.16986) and shows great promise as a flexible and scalable quantum algorithm.  

To explore more features of GQE, check out the CUDA-Q Solvers [documentation](https://nvidia.github.io/cudaqx/examples_rst/solvers/gqe.html).


# Implementing and Parallelizing ADAPT-VQE


---
## ADAPT-VQE Algorithm

The variational quantum eigensolver (VQE) is a well known quantum chemistry technique, which estimates the ground state of a molecule by optimizing a parameterized ansatz through iterative loops involving expectation values computed on a QPU and classical parameter updates performed by a supercomputer.  VQE suffers from a a number of issues including costly sampling of the Hamiltonian terms, and the so called **barren plateau** problem where even modest problems sizes do not converge due to the flat optimization landscape.

For VQE to have any viability for near term applications, clever implementations are necessary to avoid these challenges. One promising approch is the **Adaptive Derivative-Assembled Pseudo-Trotter VQE (ADAPT-VQE)** approach. The core idea is rather than have a fixed **ansatz**, to "adaptively" build the ansatz over the course of the algorithm by performing the following steps:

1. On classical hardware, compute the one and two-electron integrals and transform the Hamiltonian to a qubit representation.
2. Build an **operator pool**, usually corresponding to physical excitations from techniques like **unitary coupled cluster with singles and doubles (UCCSD)**.
3. Initialize qubits in a reference Hatree Fock state.
4. Measure the **commutator** of the Hamiltonian with each operator in the operator pool to compute the gradient.
5. If the norm of the gradient is below a threshold, then exit.  Otherwise, add the operator with the largest gradient contribution along with a new variational parameter, $\theta_{i+1}$.
6. Run VQE on this circuit to obtain optimized variational parameters.
7. Repeat steps 4 through 6 until convergence.

All of the steps are represented below in the following figure.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/adapt_workflow.png?raw=1" alt="Flowchart of the seven-step ADAPT-VQE algorithm: classical Hamiltonian computation, UCCSD operator pool construction, Hartree-Fock initialization, commutator-based gradient measurement, convergence check, VQE optimization with the new operator, and iteration until convergence." title="ADAPT-VQE Workflow" width="1300">


If you are a careful reader, this may seem like a we are now running an entire VQE experiment for each iteration.  This is correct, but, the additive constructions of the ansatz means that we begin with very simple and easy to optimize circuits relative to a starting a VQE procedure with a fixed ansatz with many circuit parameters. Using the gradient allows strategic operator selection so we are optimizing parameters for operators more likely to converge to the ground state. Adding new parameters one at a time to an already converged solution allows each iteration to begin with a warm start.  In practice, ADAPT-VQE generally converges faster than VQE and is more resilient to barren plateaus given this warm start.  

To get a better sense of the process for ADAPT-VQE and how it compares with VQE, try the following interactive widget linked [here](https://nvidia.github.io/cuda-q-academic/chemistry-simulations/Images/adapt_vqe/adapt_widget.html).  The data is artificial, but is qualitatively representative of the two approaches.

The original paper for ADAPT-VQE entitled ["An adaptive variational algorithm for exact molecular simulations on a quantum computer"](https://www.nature.com/articles/s41467-019-10988-2) presents data (figure below) that demonstrates the performance benefits of the ADAPT approach.  The chart below shows the convergence of different ansatz construction techniques compared to ADAPT as a function of variational parameters for BeH.  ADAPT converges much faster and with far fewer parameters.  

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/adapt_convergence.png?raw=1" alt="Line plot from the original ADAPT-VQE paper comparing energy convergence of ADAPT versus UCCSD and other fixed ansatze for BeH as a function of the number of variational parameters, showing ADAPT requiring fewer parameters to reach near-exact ground state energy." title="ADAPT-VQE Convergence" width="500">


The standard UCCSD ansatz is noted in the plot as having nearly twice the number of parameters as ADAPT as well as much worse convergece.

The rest of this lab will walk you though coding up an ADAPT-VQE implementation.  You will explore the impact of different operator pools as well as learn how ADAPT is well suited for parallelization across multiple QPUs.


---

## Building the Operator Pool and Computing the Gradient

Before building the entire workflow, it is helpful to construct some of the subroutines. You will begin by creating subroutines to define the operator pool and to compute the gradient.  First, consider the operator pool. Any Pauli word could be placed in the operator pool. An operator $A_i$ can then be selected and applied to a quantum circuit as $e^{-i\theta_iA_i}$ to build a parameterized quantum circuit ansatz. For traditional VQE, the same procedure is used, except all of the operators are applied *a priori* to construct a fixed parametrized ansatz.

Naively one might think that a bigger and more diverse operator pool is inherently better. However, because quantum chemistry has certain symmetries, many operators would be poor choices and even violate physical constraints of the system.

A common choice of operators ubiquitous in chemistry  comes from unitary coupled cluster with single and double excitations (UCCSD). Where single excitations are defined as:

$$ T_{ij} = \frac{i}{2}(X_iY_j - Y_iX_j)\prod_{p=i+1}^{j-1} Z_p $$

And double excitations as:

$$ T_{ijkl} = \frac{i}{8}(X_iY_jX_kX_l +Y_iX_jX_kX_l + Y_iY_jY_kX_l + Y_iY_jX_kY_l - X_iX_jY_kX_l - X_iX_jX_kY_l - Y_iX_jY_kY_l - X_iY_jY_kY_l) \prod_{p=i+1}^{j-1} Z_p \prod_{r=k+1}^{l-1} Z_r $$





### Exercise 7:

This exercise will allow you to explore some of the reasons why poor selection of an operator pool can lead to convergence issues with ADAPT-VQE.

We've demonstrated how to create an $H_2$ molecule using $\texttt{solvers}$.  Your task is to
prepare a Hartree Fock kernel.
The second part of this exercise is to write one more kernel that applies $T_{0123}$ to the Hartree Fock state. To help guide you, we've created a kernel that applies  $Z_0Z_1Z_2Z_3$ as an example using $\texttt{exp\_pauli}$ where the coefficient is a variational parameter.

Run VQE and comment on the results.

In [ ]:
# EXERCISE 7
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, casci=True)

norbitals = molecule.n_orbitals
nelectrons = molecule.n_orbitals
print(molecule.energies)
print(molecule.hamiltonian)


@cudaq.kernel # HF kernel
def kernel(theta: List[float]):
    reg = cudaq.qvector(4)

    ##TODO## initialize the first two qubits in the one-state (x gates)


@cudaq.kernel # T0123 kernel
def kernel_a(theta: List[float]):
    reg = cudaq.qvector(4)

    ##TODO## initialize the first two qubits in the one-state, then apply T0123



@cudaq.kernel # Z0Z1Z2Z3 example kernel
def kernel_b(theta: List[float]):
    reg = cudaq.qvector(4)

    x(reg[0])
    x(reg[1]) # Initialize the first two qubits in the one-state
    exp_pauli(theta[0], reg, "ZZZZ")


kernel_list = [kernel, kernel_a, kernel_b]


for k in kernel_list:
    cost_values = []

    def cost(parameters):

        expectation_value = cudaq.observe(k, molecule.hamiltonian,
                                          parameters).expectation()
        cost_values.append(expectation_value)

        return expectation_value

    initial_parameters = [0.5]
    initial_cost_value = cost(initial_parameters)
    print("Initial Energy:", initial_cost_value)


    result = minimize(cost,
                      initial_parameters,
                      method='Nelder-Mead', options={'xatol':1e-8, 'fatol':1e-12, 'maxiter':300})

    print("Energy:", result.fun)


    plt.figure(figsize=(4, 3))

    x_values = list(range(len(cost_values)))
    y_values = cost_values

    plt.plot(x_values, y_values)
    plt.xlabel("Epochs")
    plt.ylabel("Cost Value")
    plt.grid(True)

    # Show the plot
    plt.show()


    print(result.x)

The reason the $Z_0Z_1Z_2Z_3$ operator did not work, is that it commutes with the Hamiltonian. This means that it has no impact on the expectation value and is therefore a useless addition to an operator pool, potentially adding to the circuit depth with no gain in accuracy. Confirm this below by computing its commutator with the Hamiltonian:

$$ [H,A] = HA -AH $$

In [ ]:
zzzz_operator = spin.z(0)*spin.z(1)*spin.z(2)*spin.z(3)
print("Commutator:", molecule.hamiltonian*zzzz_operator - zzzz_operator*molecule.hamiltonian)

Another potential issue is adding an operator that breaks a symmetry like the number of electrons.  Even if such an operator lowered the energy, you no longer have a solution for the molecule in question.  

Use the number operator $N = 2*I - \frac{1}{2}(Z_0 + Z_1 + Z_2 + Z_3)$ to compute $\bra{\psi}N\ket{\psi}$ and prove that $Z_0Z_1X_2 +Z_0Z_1Y_2$ produces states that have more than 2 electrons.

In [ ]:
number_operator = 2 * spin.i(0)*spin.i(1)*spin.i(2)*spin.i(3) - 0.5*spin.z(0) - 0.5*spin.z(1) - 0.5*spin.z(2) - 0.5*spin.z(3)

@cudaq.kernel # HF operator
def kernel():
    reg = cudaq.qvector(4)

    ##TODO## initialize the first two qubits in the one-state (x gates)


@cudaq.kernel
def kernel_c():
    reg = cudaq.qvector(4)

    ##TODO## initialize the first two qubits in the one-state, then apply Z_0Z_1X_2 + Z_0Z_1Y_2

print("Particle Number HF:", cudaq.observe(kernel, number_operator).expectation())
print("Particle Number:", cudaq.observe(kernel_c, number_operator).expectation())
# END EXERCISE

Auxiliary scripts in CUDA-Q make it easy to build a standard operator pool using a function like `get_uccsd_pool` to provide a list of spin operators corresponding to all UCCSD excitations. The code also splits each operator into a list of Pauli words (`word_pool`) and a list of the associated coefficients (`sign_pool`).   

This section of code also includes a function to compute the commutator operator for each element in the full operator pool. Note, that the factor of `i` was removed from each operator to make data transfer into CUDA-Q kernels easier, but it is included in the commutator function below for accuracy.

In [ ]:
from aux_files.classical_pyscf import get_mol_hamiltonian
from aux_files.operator_pool import get_uccsd_pool
n_qubits= norbitals * 2

pools,word_pool,sign_pool = get_uccsd_pool(nelectrons, n_qubits)

print('Number of operator pool: ', len(pools))
print('Word Pool: ', word_pool)
print('Sign Pool: ', sign_pool)


def commutator(pools, ham):
    com_op = []

    for i in range(len(pools)):
        # We add the imaginary number that we excluded when generating the operator pool.
        op = 1j * pools[i]

        com_op.append(ham * op - op * ham)

    return com_op

grad_op = commutator(pools, molecule.hamiltonian)

---

## Preparing $\ket{\psi}$ Kernels

So far, you have completed a number of the prerequisite functions required to construct the ADAPT-VQE workflow. The figure below shows green checks for these already completed items.  You will now focus on the yellow boxes where a state $\ket{\psi}$ is prepared from which the gradient $\bra{\psi}[H,A]\ket{\psi}$ is computed.  

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/workflow_middle.drawio.png?raw=1" alt="ADAPT-VQE workflow diagram with green checkmarks on completed steps (Hamiltonian construction, operator pool, commutator definition) and yellow boxes highlighting the upcoming psi state preparation and gradient evaluation steps." title="ADAPT-VQE Workflow Progress" width="1300">


Working backwards, the simple kernel below takes advantage of CUDA-Q's state passing capabilites to build a kernel constructed from an arbitrary state vector.  Such an abstraction is helpful as it allows easy separation from an initial state kernel (the Hartree Fock state) and the resulting kernel from each ADAPT-VQE iteration.  Regardless of the origin, `cudaq.get_state()` can be used to extract the state vector which will be input to the kernel `psi` to initialize the next iteration.

In [ ]:
@cudaq.kernel
def psi(state:cudaq.State):
    q = cudaq.qvector(state)

### Exercise 8:

You will now construct the two sorts of kernels which will be used to construct input states for $\texttt{psi}$.  
        
1. A kernel which prepares the Hartree Fock state
   
2. A kernel which applies all operators selected from the previous ADAPT-VQE iterations on the Hartree Fock state.

The second kernel is the tricky one here as the kernel must receive a number of inputs in order to apply the selected operators and their optimized $\theta$'s. CUDA-Q kernels require inputs to be simple typed lists consisting of strings, integers, or floats.  This may seem inconvenient, but is an important aspect of ensuring kernels are lightweight and result in high performance computations.  Thus, it is easiest to break the input operators into separate lists corresponding to the coefficients, and Pauli words of the singles and doubles operators, respectively.    

The kernel definition is provided for you below.  Fill in the appropriate code to apply all of the operators selected from the ADAPT-VQE process. Hints:
* Consider two separate loops, one for applying the singles operators and one for applying the doubles, each starting from a flattened list of all operators.
* Assume the theta list contains the parameters corresponding to the singles operators followed by the doubles operators sorted by the order in which they were added.
* Also, do not forget to initialize the Hartree Fock state.


In [ ]:
# EXERCISE 8
# Write a kernel to prepare the HF reference state
@cudaq.kernel
def initial_state(n_qubits:int, nelectrons:int):
        ##TODO## allocate qubits and initialize the Hartree-Fock reference state


state = cudaq.get_state(initial_state, n_qubits, nelectrons)
print(state)

In [ ]:
@cudaq.kernel
def kernel(theta: list[float], qubits_num: int, nelectrons: int, pool_single: list[cudaq.pauli_word],
           coef_single: list[float], pool_double: list[cudaq.pauli_word], coef_double: list[float]):

    ##TODO## initialize HF state, then apply singles and doubles from selected pool operators

---

## Putting it all Together




### Exercise 9:

You are now ready to put all of your work together and construct the complete ADAPT-VQE workflow. For this exercise, you will fill in the incomplete parts of the code (the yellow boxes in the figure below) and then run your code to compute molecular energies.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-4-images/workflow_end.png?raw=1" alt="ADAPT-VQE workflow diagram with yellow boxes highlighting the four remaining implementation tasks: gradient vector computation, operator selection, VQE cost function definition, and state update after optimization." title="Final Steps of ADAPT-VQE Workflow" width="1300">


First, scan the entire cell below and see if the rough workflow makes sense.

There are four "TODO" sections. Complete the following tasks where annotated below:
1. Compute the commutator $[H,A]$ for each term in the operator pool, store as the gradient vector, and check if the gradient norm has converged.
2. Identify and print the operator selected at step $i$, adding it to the list of selected operators.
3. Define the cost function for the intermediate VQE optimization that includes the kernel with the new operator added.
4. Save the updated energy and compute a new trial state $\ket{\psi}$ with the optimized parameters

By completing these four central tasks, you should be able to run ADAPT-VQE and compute the energy of the $H_2$ ground state. How does it compare to the classically computed FCI energy?




In [ ]:
# EXERCISE 9
#geometry = [('O', (0., 0., 0.)), ('H', (-0.75, 0.5, 0.0)), ('H', (-0.75, -0.5, 0.0))]
#geometry = [('Li', (0., 0., 0.)), ('H', (0., 0., 1.6))]
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, ccsd=True, casci=True)

norbitals = molecule.n_orbitals
nelectrons = molecule.n_electrons
n_qubits= norbitals * 2

print(molecule.energies)

#Pulling commands from above so you can edit molecule and run this cell only
pools,word_pool,sign_pool = get_uccsd_pool(nelectrons, n_qubits)
grad_op = commutator(pools, molecule.hamiltonian)
state = cudaq.get_state(initial_state, n_qubits, nelectrons)


print('Beginning of ADAPT-VQE')

threshold=1e-3
E_prev=0.0
e_stop=1e-5
init_theta=0.0

energies_adapt=[]


theta_single=[]
theta_double=[]

pool_single=[]
pool_double=[]

coef_single=[]
coef_double=[]

selected_pool=[]

for step in range(5):

    print('Step: ', step)
    start_time = time.time()
    # SECTION 1: Compute the gradient w.r.t each operator in the pool.
    #also compute the gradient norm and decide if ADAPT-VQE has converged or not



    # SECTION 1: END

    if norm <= threshold:
        print('\n', 'Final Result: ', '\n')
        print('Final parameters: ', theta)
        print('Selected pools: ', selected_pool)
        print('Number of pools: ', len(selected_pool))
        print('Final energy: ', result_vqe.fun)

        break

    else:




        # SECTION 2:  Find the operator corresponding to the largest gradient element.
        #Add this operator and its coefficient to the appropriate single or doubles list.



        # SECTION 2: End

        print('Operator selected at current step: ', temp_op)

#-----------------------------------------------------------------------
        #This code updates the operator lists with the newly selected operator
        #and adds a new theta parameter
        tot_single=0
        tot_double=0
        for p in temp_op:
            if len(p) == 2:
                tot_single += 1
                for word in p:
                    pool_single.append(word)
            else:
                tot_double += 1
                for word in p:
                    pool_double.append(word)

        for coef in temp_sign:
            if len(coef) == 2:
                for value in coef:
                    coef_single.append(value.real)
            else:
                for value in coef:
                    coef_double.append(value.real)

        print('pool single: ', pool_single)
        print('coef_single: ', coef_single)
        print('pool_double: ', pool_double)
        print('coef_double: ', coef_double)
        print('tot_single: ', tot_single)
        print('tot_double: ', tot_double)

        init_theta_single = [init_theta] * tot_single
        init_theta_double = [init_theta] * tot_double

        theta_single = theta_single + init_theta_single
        theta_double = theta_double + init_theta_double

        print('theta_single', theta_single)
        print('theta_double: ', theta_double)

        theta = theta_single + theta_double
        print('theta', theta)
#------------------------------------------------------------------


        # SECTION 3: define a cost function to optimize the energy of the circuit with the newly selected operator
        def cost(theta: list[float]) -> float:
            ##TODO## compute expectation value using cudaq.observe(kernel, ...) and return energy

        # SECTION 3: END

        # Compute the gradient using parameter shift
        def parameter_shift(theta):
            parameter_count = len(theta)
            grad = np.zeros(parameter_count)
            theta2 = theta.copy()
            for i in range(parameter_count):
                theta2[i] = theta[i] + np.pi/4
                exp_val_plus = cost(theta2)
                theta2[i] = theta[i] - np.pi/4
                exp_val_minus = cost(theta2)
                grad[i] = (exp_val_plus - exp_val_minus)
                theta2[i] = theta[i]
            return grad



#-------------------------------------------------------------------
        #This section saves the optimized parameters, prints the energy and determines if the energy has converged.
        result_vqe=minimize(cost, theta, method='BFGS', jac=parameter_shift,  options={'maxiter':100})

        theta=result_vqe.x.tolist()
        theta_single = theta[:tot_single]
        theta_double = theta[tot_single:]

        print('Optmized Energy: ', result_vqe.fun)
        print('Optimizer exited successfully: ',result_vqe.success, flush=True)
        print(result_vqe.message, flush=True)

        energies_adapt.append(result_vqe.fun)
        dE= result_vqe.fun-E_prev
        print('dE: ', dE)
        print('\n')
        end_time = time.time()
        print("Step Time:", end_time -start_time)

        if np.abs(dE)<=e_stop:
            print('\n', 'Final Result: ', '\n')
            print('Final parameters: ', theta)
            print('Selected pools: ', selected_pool)
            print('Number of pools: ', len(selected_pool))
            print('Final energy: ', result_vqe.fun)

            break
#-----------------------------------------------------------------
        else:

            # SECTION 4: - Update the energy for the next iteration
            # Update the state variable to correspond to the optimized circuit with the newly added operator
            E_prev=result_vqe.fun

            state = cudaq.get_state ##TODO## call get_state with kernel and optimized theta

            # SECTION 4: END



### Exercise 10:

Now that you have a working version of ADAPT-VQE, explore how it matches up to standard VQE.  Go back to the previous cell and change the molecule to LiH. Run ADAPT-VQE for a max of 5 iterations.  Then, run standard VQE using the cell below and plot the convergence of the two data series.  How do the convergence behaviors compare?  How many parameters does standard VQE need to optimize?  What is the maximum number of parameters that require optimization for your ADAPT-VQE run with LiH?



In [ ]:
# EXERCISE 10
n_theta = solvers.stateprep.get_num_uccsd_parameters(nelectrons, n_qubits, spin=0)
theta_0 = np.random.uniform(-0.2, 0.2, size=n_theta)
print("Number of VQE parameters:", n_theta)

vqe_energies = []

@cudaq.kernel
def vqe_kernel(qubit_num: int, n_electrons: int, theta: list[float]):

    q = cudaq.qvector(qubit_num)

    for i in range(nelectrons):
        x(q[i])

    solvers.stateprep.uccsd(q, theta, n_electrons,0)


def cost(thetas: np.ndarray) -> float:

    energy=cudaq.observe(vqe_kernel, molecule.hamiltonian, n_qubits, nelectrons, thetas).expectation()
    return energy

def cb(xk: np.ndarray) -> None:
    e = cost(xk)
    vqe_energies.append(e)
    print(e)


result = minimize(cost, theta_0,  method='BFGS', jac='2-point', callback=cb,  options={'maxiter':5} )

In [ ]:
x_vqe   = np.arange(len(vqe_energies))
x_adapt = np.arange(len(energies_adapt))

plt.figure(figsize=(6, 4))
plt.plot(x_vqe,   vqe_energies,   '^-', label='VQE')
plt.plot(x_adapt, energies_adapt, 'o-', label='ADAPT-VQE')

plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.xlim(0, len(vqe_energies) - 1)
plt.legend()
plt.tight_layout()
plt.show()

---

## Parallelizing ADAPT-VQE

In the future, data centers will host multiple QPUs, allowing quantum algorithms to parallelize and run faster. For example, obtaining one million circuit samples can be accomplished by pooling 100,000 samples from 10 QPUs.

ADAPT-VQE is well suited to benefit from parallelization in a few parts of the code. In the next exercise, you will explore how this can speed up the runtime of ADAPT-VQE.

Computing the commutators for the gradient in each ADAPT-VQE step is a trivially parallelizable task.  Each operator is independent of the others, so each one can be evaluated on a separate QPU.  CUDA-Q's MQPU backend allows you to simulate this by designating each circuit to run on a different virtual QPU simulated by a GPU.

Examine the code below to see how this is done.  Simply change the backend with `cudaq.set_target('nvidia', option='mqpu,fp64')` and `cudaq.observe` with `cudaq.observe_async` and add a variable `qpu_id` which specifies an integer corresponding to your index of available GPUs. These results are run asynchronously and stored in a list of futures which are then accessed later with `get()`.  One slight complication is when the initial set and subsequent states are obtained with `get_state`, they must also be updated to `get_state_async`.  You can no longer save the new state at the end of each new iteration as the `get_state_async` needs to run on the same `qpu_id` as the command computing the expectation value, otherwise the state will be prepared on a different GPU.




### Exercise 11:

Your task is to further parallelize the code and modify the $\texttt{parameter\_shift}$ function, which computes the gradient of the VQE step, to run asynchronously across two simulated QPUs.  Use the parallel code for the commutator evaluation as a guide for this.

Next, add flags to capture the execution time of each step of the the serial and parallel implementations.  Then, run them on LiH for a maximum of 5 steps (iterations).  Do you notice a difference?  The longer ADAPT-VQE runs, the harder each circuit evaluation and VQE procedure becomes, meaning there is greater benefit from using parallelization.  The benefit may be small in this case as the circuits are not very large. This is related to the to the fact that a GPU must be used for a problem large enough to benefit from parallelization. Asynchronous execution across multiple QPUs will be most valuable when the individual tasks are harder.


In [ ]:
# EXERCISE 11
cudaq.set_target('nvidia', option='mqpu,fp64')

#geometry = [('O', (0., 0., 0.)), ('H', (-0.75, 0.5, 0.0)), ('H', (-0.75, -0.5, 0.0))]
#geometry = [('Li', (0., 0., 0.)), ('H', (0., 0., 1.6))]
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, ccsd=True, casci=True)

norbitals = molecule.n_orbitals
nelectrons = molecule.n_electrons
n_qubits= norbitals * 2

print(molecule.energies)

#Pulling commands from above so you can edit molecule and run this cell only
pools,word_pool,sign_pool = get_uccsd_pool(nelectrons, n_qubits)
grad_op = commutator(pools, molecule.hamiltonian)


print('Beginning of ADAPT-VQE')

threshold=1e-3
E_prev=0.0
e_stop=1e-5
init_theta=0.0

energies_adapt.append(molecule.energies['hf_energy'])


theta_single=[]
theta_double=[]

pool_single=[]
pool_double=[]

coef_single=[]
coef_double=[]

selected_pool=[]

max_steps = 5

for step in range(max_steps):

    print('Step: ', step)
    start_time = time.time()

#-------------------------------------------------------------------
    async_results =[]
    async_states=[]
    #Load state on each GPU
    for gpu_id in range(2): #2 gpus
        if step == 0:
            async_states.append(cudaq.get_state_async(initial_state, n_qubits, nelectrons, qpu_id = gpu_id).get())
        else:
            async_states.append(cudaq.get_state_async(kernel, theta, n_qubits, nelectrons,
                                                      pool_single,coef_single, pool_double, coef_double, qpu_id=gpu_id).get())

    for i in range(len(grad_op)):
        qpu_id = i % 2 #paralellize across 2 GPUs

        async_results.append(cudaq.observe_async(psi, grad_op[i], async_states[qpu_id], qpu_id = qpu_id))

    gradient_vec = [res.get().expectation() for res in async_results]

#-------------------------------------------------------------------

    norm=np.linalg.norm(np.array(gradient_vec))
    print('Norm of the gradient: ', norm)



    if norm <= threshold:
        print('\n', 'Final Result: ', '\n')
        print('Final parameters: ', theta)
        print('Selected pools: ', selected_pool)
        print('Number of pools: ', len(selected_pool))
        print('Final energy: ', result_vqe.fun)

        break

    else:



        max_grad=np.max(np.abs(gradient_vec))
        print('max_grad: ', max_grad)

        temp_op = []
        temp_sign = []
        for i in range(len(word_pool)):
            if np.abs(gradient_vec[i]) == max_grad:
                temp_op.append(word_pool[i])
                temp_sign.append(sign_pool[i])

        selected_pool = selected_pool + temp_op

        print('Operator selected at current step: ', temp_op)


        tot_single=0
        tot_double=0
        for p in temp_op:
            if len(p) == 2:
                tot_single += 1
                for word in p:
                    pool_single.append(word)
            else:
                tot_double += 1
                for word in p:
                    pool_double.append(word)

        for coef in temp_sign:
            if len(coef) == 2:
                for value in coef:
                    coef_single.append(value.real)
            else:
                for value in coef:
                    coef_double.append(value.real)

        print('pool single: ', pool_single)
        print('coef_single: ', coef_single)
        print('pool_double: ', pool_double)
        print('coef_double: ', coef_double)
        print('tot_single: ', tot_single)
        print('tot_double: ', tot_double)

        init_theta_single = [init_theta] * tot_single
        init_theta_double = [init_theta] * tot_double

        theta_single = theta_single + init_theta_single
        theta_double = theta_double + init_theta_double

        print('theta_single', theta_single)
        print('theta_double: ', theta_double)

        theta = theta_single + theta_double
        print('theta', theta)

        def cost(theta):

            theta=theta.tolist()

            energy=cudaq.observe(kernel, molecule.hamiltonian, theta, n_qubits, nelectrons, pool_single,
                                coef_single, pool_double, coef_double).expectation()

            return energy

        ##TODO## Update the gradient computation to parallelize across two GPUs
        # Compute the gradient using parameter shift
        def parameter_shift(theta):
            parameter_count = len(theta)
            grad = np.zeros(parameter_count)
            theta2 = theta.copy()
            for i in range(parameter_count):
                theta2[i] = theta[i] + np.pi/4
                exp_val_plus = cudaq.observe_async(kernel, molecule.hamiltonian, theta2, n_qubits, nelectrons, pool_single,
                                coef_single, pool_double, coef_double, qpu_id=##TODO##)
                theta2[i] = theta[i] - np.pi/4
                exp_val_minus = cudaq.observe_async(kernel, molecule.hamiltonian, theta2, n_qubits, nelectrons, pool_single,
                                coef_single, pool_double, coef_double, qpu_id=##TODO##)
                grad[i] = (exp_val_plus.get().expectation() - exp_val_minus.get().expectation())
                theta2[i] = theta[i]
            return grad
        # END EXERCISE


#-------------------------------------------------------------------
        #This section saves the optimized parameters, prints the energy and determines if the energy has converged.
        result_vqe=minimize(cost, theta, method='BFGS', jac=parameter_shift,  options={'maxiter':100})

        theta=result_vqe.x.tolist()
        theta_single = theta[:tot_single]
        theta_double = theta[tot_single:]

        print('Optmized Energy: ', result_vqe.fun)
        print('Optimizer exited successfully: ',result_vqe.success, flush=True)
        print(result_vqe.message, flush=True)

        energies_adapt.append(result_vqe.fun)
        dE= result_vqe.fun-E_prev
        print('dE: ', dE)
        print('\n')
        end_time = time.time()
        print("Step Time:", end_time - start_time)

        if np.abs(dE)<=e_stop:
            print('\n', 'Final Result: ', '\n')
            print('Final parameters: ', theta)
            print('Selected pools: ', selected_pool)
            print('Number of pools: ', len(selected_pool))
            print('Final energy: ', result_vqe.fun)

            break

        else:

            E_prev=result_vqe.fun

---

## Conclusion

At this point, you have coded an ADAPT-VQE implementation of your own and explored specific parts of the workflow in detail like operator pool selection. There is great benefit in understanding the details, but often it is helpful to have an out-of-the-box implementation that is optimized and easy to plug into your larger workflow or ready to generate data for your research.  CUDA-Q Solvers provides easy to use ADAPT-VQE functions so you can produce the same results without the need to code at the kernel level.

The code below, with just a few lines, runs the entire procedure coded above. You can easily swap out operator pools, molecules, and still run in parallel by following the instructions [outlined in the docs](https://nvidia.github.io/cudaqx/examples_rst/solvers/adapt.html).


In [ ]:
cudaq.set_target('nvidia', option = 'fp64')
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., .7474))]
molecule = solvers.create_molecule(geometry, 'sto-3g', 0, 0, casci=True)

numElectrons = molecule.n_electrons

operators = solvers.get_operator_pool("spin_complement_gsd",num_orbitals=molecule.n_orbitals)

@cudaq.kernel
def initState(q: cudaq.qview):
    for i in range(numElectrons):
        x(q[i])

energy, thetas, ops = solvers.adapt_vqe(initState, molecule.hamiltonian,
                                        operators)

print("<H> = ", energy)

Though ADAPT methods have clear benefits over conventional variational algorithms, they still suffer from the same fundamental measurements problems and barren plateaus for large problem sizes. Nevertheless, they are fantastic drop-in substitutions for any situation where a variation method is used for a subroutine like state preparation.

Generalizing beyond ADAPT methods, with this notebook you have learned how to use fundamental CUDA-Q functions like `exp_pauli`, `get_state_async` and `observe_async` as well as how to use a CUDA-Q Solvers method out-of-the box.  To learn more, visit the [CUDA-Q docs page](https://nvidia.github.io/cuda-quantum/latest/index.html) and the [CUDA-Q Solvers docs](https://nvidia.github.io/cudaqx/examples_rst/solvers/examples.html).
